<a href="https://colab.research.google.com/github/erdenebayrd/mit/blob/main/sEMG_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Sep  7 02:15:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!git submodule update --init text_alignments
!tar -xzf text_alignments/text_alignments.tar.gz
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
Cloning into 'silent_speech'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 518 (delta 210), reused 263 (delta 138), pack-reused 156 (from 1)
Receiving objects: 100% (518/518), 2.61 MiB | 22.83 MiB/s, done.
Resolving deltas: 100% (279/279), done.
/content/silent_speech
Submodule 'text_alignments' (https://github.com/dgaddy/silent_speech_alignments.git) registered for path 'text_alignments'
Cloning into '/content/silent_speech/text_alignments'...
Submodule path 'text_alignments': checked out '5c71ae9fcbb94e74e19eb9547c3b404baf6126a7'


In [ ]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy huggingface_hub safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 148.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 157.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 160.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
%env DATA_PATH=/content/data

env: DATA_PATH=/content/data


In [ ]:
%cd /content/silent_speech
!mkdir -p /content/data/Gaddy/h5
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/ && echo "h5 copied" || echo "!! h5 NOT on Drive"
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/ && echo "KenLM copied" || echo "!! KenLM NOT on Drive"
!cp /content/drive/MyDrive/silent_speech/emg_data.tar.gz /content/data/Gaddy/ 2>/dev/null && echo "tar restored" || echo "no tar on Drive; downloading fresh"
!python download_data.py

/content/silent_speech
h5 copied
KenLM copied
tar restored
2026-09-07 02:22:28.383930: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-07 02:22:28.401323: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788747748.422619    2772 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788747748.429126    2772 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-07 02:22:28.450703: I tensorflow/core/platform/cpu_feature_guard.cc:210] Thi

In [ ]:
import os, torch
os.chdir("/content/silent_speech")
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from architecture import EMGTransformer
from data_utils import TextTransform

st = hf_hub_download("MatteoFasulo/TinyMyo", "pretraining/TinyMyo/TinyMyo.safetensors")
sd = load_file(st)
clean = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in sd.items()}
torch.save({"state_dict": clean}, "/content/drive/MyDrive/silent_speech/TinyMyo_backbone.pt")

n_chars = len(TextTransform().chars)
m = EMGTransformer(num_features=8, num_outs=n_chars+1, in_chans=8,
                   embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4)
missing, unexpected = m.load_state_dict(clean, strict=False)
blk = sum(1 for k in m.state_dict() if k.startswith("blocks.") and k not in missing)
print("backbone tensors:", len(sd))
print("transformer-block tensors loaded:", blk, "(want ~104 for 8 layers)")
print("missing (train fresh):", len(missing), "->", list(missing)[:6])
print("unexpected (ignored):", len(unexpected), "->", list(unexpected)[:6])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pretraining/TinyMyo/TinyMyo.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

backbone tensors: 104
transformer-block tensors loaded: 96 (want ~104 for 8 layers)
missing (train fresh): 120 -> ['conv_blocks.0.conv1.weight', 'conv_blocks.0.conv1.bias', 'conv_blocks.0.norm1.weight', 'conv_blocks.0.norm1.bias', 'conv_blocks.0.norm1.running_mean', 'conv_blocks.0.norm1.running_var']
unexpected (ignored): 8 -> ['channel_embed', 'mask_token', 'model_head.decoder_pred.bias', 'model_head.decoder_pred.weight', 'norm.bias', 'norm.weight']


In [ ]:
import json, os
p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg["num_layers"]          = 8   # MUST match the 8-layer backbone
cfg["start_training_from"] = "/content/drive/MyDrive/silent_speech/TinyMyo_backbone.pt"
cfg["ckpt_directory"]      = "/content/drive/MyDrive/silent_speech/output_finetune"
cfg["num_epochs"]          = 60
cfg["eval_interval"]       = 5
cfg["num_workers"]         = os.cpu_count()
json.dump(cfg, open(p, "w"), indent=4)
print("fine-tune config ready")

fine-tune config ready


In [ ]:
import os
def chk(l, path): print(f"{l:16}: {'OK' if os.path.exists(path) else 'MISSING <- fix this'}")
chk("h5",         "/content/data/Gaddy/h5/emg_dataset.h5")
chk("raw voiced", "/content/data/Gaddy/emg_data/voiced_parallel_data")
chk("KenLM lm",   "/content/silent_speech/KenLM/lm.bin")
chk("lexicon",    "/content/silent_speech/KenLM/gaddy_lexicon.txt")
chk("backbone",   "/content/drive/MyDrive/silent_speech/TinyMyo_backbone.pt")

h5              : OK
raw voiced      : OK
KenLM lm        : OK
lexicon         : OK
backbone        : OK


In [ ]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-09-07 02:24:50.455141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788747890.475782    3475 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788747890.482034    3475 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
['recognition_model.py']
output example: (/content/data/Gaddy/emg_data/silent_parallel_data/5-11_silent, '102')
train / dev split: 8399 30
Layer (type:depth-idx)                                            Output Shape              Param #
EMGTransformer                                                    [1, 200, 38]              --
├─Sequential: 1-1                                                 [1, 192, 200] 

In [ ]:
import os, glob
os.chdir("/content/silent_speech")
os.environ["BEST"] = sorted(glob.glob("/content/drive/MyDrive/silent_speech/output_finetune/model_*_best.pt"), key=os.path.getmtime)[-1]
print("evaluating:", os.environ["BEST"])
!python recognition_model.py --evaluate_saved "$BEST"